In [1]:
# Setup & Data Ingestion Engine (Imports & Files Loading)
import pandas as pd
import json
import numpy as np

print("="*60)
print("STEP 1: LOADING RAW DATASETS (JSON & CSVs)")
print("="*60)

# 1. Load JSON Specs & Details
with open("dawlance_exact_product_data.json", "r", encoding="utf-8") as f:
    dawlance_json = json.load(f)

with open("haier_19lf_daraz_full_details.json", "r", encoding="utf-8") as f:
    haier19_json = json.load(f)

with open("haier_pearl_black_hsu18hfp_full_details.json", "r", encoding="utf-8") as f:
    haier_pearl_json = json.load(f)

# 2. Load CSV Specs & Features
df_dawlance_specs = pd.read_csv("dawlance_exact_product_specs.csv")
df_haier19_specs = pd.read_csv("haier_19lf_daraz_specs.csv")
df_haier_pearl_specs = pd.read_csv("haier_pearl_black_specs.csv")

# 3. Load Customer Reviews Datasets
r_dawlance_mega = pd.read_csv("dawlance_mega_t10_200_pages_reviews.csv")
r_dawlance_official = pd.read_csv("dawlance_official_cleaned_reviews.csv")
r_haier19 = pd.read_csv("haier_reviews.csv")
r_haier_pearl = pd.read_csv("haier_pearl_black_hsu18hfp_reviews.csv")

print(f"[SUCCESS] Dawlance Mega Reviews: {len(r_dawlance_mega)} rows")
print(f"[SUCCESS] Haier 19LF Reviews: {len(r_haier19)} rows")
print(f"[SUCCESS] Haier Pearl Black Reviews: {len(r_haier_pearl)} rows")

STEP 1: LOADING RAW DATASETS (JSON & CSVs)
[SUCCESS] Dawlance Mega Reviews: 799 rows
[SUCCESS] Haier 19LF Reviews: 49 rows
[SUCCESS] Haier Pearl Black Reviews: 18 rows


Specs & Features Cleaning & Standardization Model

In [2]:
print("="*60)
print("STEP 2: RUNNING TECHNICAL SPECS STANDARDIZATION MODEL")
print("="*60)

# Build Normalized Master Comparison Matrix
specs_data = [
    {
        "Product_ID": "DW-MEGAT10",
        "Brand": "Dawlance",
        "Model": "Mega T+ Inverter 10 CO",
        "Capacity_Ton": 0.75,
        "BTU_Rating": 9000,
        "Price_PKR": 78999,
        "Inverter_Technology": "DC Inverter",
        "Condenser_Type": "Gold Fin Anti-Rust",
        "Compressor_Warranty_Years": 12,
        "PCB_Warranty_Years": 4,
        "Key_Feature": "4 Gen Energy Saver & i-Clean",
        "Ideal_Room_SqFt": 150
    },
    {
        "Product_ID": "HR-HSU19LF",
        "Brand": "Haier",
        "Model": "HSU-19LF (White)",
        "Capacity_Ton": 1.5,
        "BTU_Rating": 19000,
        "Price_PKR": 134999,
        "Inverter_Technology": "DC Inverter (UPS Enabled)",
        "Condenser_Type": "100% Pure Copper Golden Fin",
        "Compressor_Warranty_Years": 10,
        "PCB_Warranty_Years": 4,
        "Key_Feature": "19,000 Full BTU & Self Cleaning",
        "Ideal_Room_SqFt": 240
    },
    {
        "Product_ID": "HR-PEARL-BLK",
        "Brand": "Haier",
        "Model": "Pearl Black (HSU-18HFP)",
        "Capacity_Ton": 1.5,
        "BTU_Rating": 18000,
        "Price_PKR": 174900,
        "Inverter_Technology": "Smart WiFi DC Inverter",
        "Condenser_Type": "100% Pure Copper Anti-Corrosion",
        "Compressor_Warranty_Years": 10,
        "PCB_Warranty_Years": 4,
        "Key_Feature": "Black Glass Panel, WiFi & Low Amp (2.4A)",
        "Ideal_Room_SqFt": 279
    }
]

df_specs_comparison = pd.DataFrame(specs_data)
df_specs_comparison.to_csv("standardized_products_matrix.csv", index=False)

# Display Cleaned Matrix in Notebook
display(df_specs_comparison[['Brand', 'Model', 'Capacity_Ton', 'BTU_Rating', 'Price_PKR', 'Compressor_Warranty_Years', 'Key_Feature']])

STEP 2: RUNNING TECHNICAL SPECS STANDARDIZATION MODEL


,Brand,Model,Capacity_Ton,BTU_Rating,Price_PKR,Compressor_Warranty_Years,Key_Feature
0,Dawlance,Mega T+ Inverter 10 CO,0.75,9000,78999,12,4 Gen Energy Saver & i-Clean
1,Haier,HSU-19LF (White),1.50,19000,134999,10,"19,000 Full BTU & Self Cleaning"
2,Haier,Pearl Black (HSU-18HFP),1.50,18000,174900,10,"Black Glass Panel, WiFi & Low Amp (2.4A)"


Review Cleaning & Sentiment Classification Model

In [3]:
print("="*60)
print("STEP 3: RUNNING REVIEW CLEANING & SENTIMENT ANALYSIS MODEL")
print("="*60)

# Standardize column headers
r_dawlance_mega = r_dawlance_mega.rename(columns={'user': 'Author', 'date': 'Date', 'review': 'Review_Text'})
r_dawlance_official = r_dawlance_official.rename(columns={'Title': 'Subject'})
r_haier19 = r_haier19.rename(columns={'user': 'Author', 'date': 'Date', 'review': 'Review_Text'})
r_haier_pearl = r_haier_pearl.rename(columns={'user': 'Author', 'date': 'Date', 'review': 'Review_Text'})

# Assign Product Tagging
r_dawlance_mega['Product'] = "Dawlance Mega T+ 10 CO"
r_dawlance_official['Product'] = "Dawlance Mega T+ 10 CO"
r_haier19['Product'] = "Haier HSU-19LF (White)"
r_haier_pearl['Product'] = "Haier Pearl Black (HSU-18HFP)"

# Combine all datasets
reviews_master = pd.concat([
    r_dawlance_mega[['Product', 'Author', 'Date', 'Review_Text']],
    r_dawlance_official[['Product', 'Author', 'Date', 'Review_Text']],
    r_haier19[['Product', 'Author', 'Date', 'Review_Text']],
    r_haier_pearl[['Product', 'Author', 'Date', 'Review_Text']]
], ignore_index=True)

# Data Cleaning: Remove missing & duplicate reviews
reviews_master.dropna(subset=['Review_Text'], inplace=True)
reviews_master.drop_duplicates(subset=['Review_Text'], inplace=True)

# Sentiment Extraction Logic
def evaluate_sentiment(text):
    t = str(text).lower()
    pos_words = ['good', 'great', 'excellent', 'best', 'chilling', 'happy', 'superb', 'original', 'working', 'satisfied', 'nice', 'fast', 'love', 'perfect', 'recommended']
    neg_words = ['bad', 'worst', 'issue', 'problem', 'faulty', 'damage', 'slow', 'leakage', 'noise', 'waste', 'broken', 'late', 'poor', 'not good']
    
    pos_score = sum(1 for w in pos_words if w in t)
    neg_score = sum(1 for w in neg_words if w in t)

    if pos_score > neg_score:
        return 'Positive'
    elif neg_score > pos_score:
        return 'Negative'
    else:
        return 'Neutral'

reviews_master['Sentiment'] = reviews_master['Review_Text'].apply(evaluate_sentiment)
reviews_master.to_csv("cleaned_sentiment_reviews.csv", index=False)

# Display Breakdown
print("\n--- SENTIMENT SUMMARY BREAKDOWN ---")
summary_table = pd.crosstab(reviews_master['Product'], reviews_master['Sentiment'], margins=True)
display(summary_table)

STEP 3: RUNNING REVIEW CLEANING & SENTIMENT ANALYSIS MODEL

--- SENTIMENT SUMMARY BREAKDOWN ---


Sentiment,Negative,Neutral,Positive,All
Product,,,,
Dawlance Mega T+ 10 CO,51,200,372,623
Haier HSU-19LF (White),7,14,28,49
Haier Pearl Black (HSU-18HFP),0,6,12,18
All,58,220,412,690


3-Way Comparative Decision & Analytics Engine

In [4]:
print("="*60)
print("STEP 4: 3-WAY MODEL COMPARISON ENGINE")
print("="*60)

# Calculate Sentiment Metrics
sent_counts = reviews_master.groupby(['Product', 'Sentiment']).size().unstack(fill_value=0)
sent_counts['Total_Reviews'] = sent_counts.sum(axis=1)
sent_counts['Positive_Ratio_%'] = ((sent_counts['Positive'] / sent_counts['Total_Reviews']) * 100).round(1)

# Merge with Specs Matrix
final_comparison = pd.merge(df_specs_comparison, sent_counts, left_on='Model', right_on='Product', how='left')

# Recommendation Decision Metric
def generate_recommendation(row):
    if row['Capacity_Ton'] == 0.75:
        return "Best Choice for Small Bedrooms & Lowest Electricity Cost"
    elif "WiFi" in str(row['Inverter_Technology']):
        return "Best Luxury Choice (Smart App Control & Black Aesthetic)"
    else:
        return "Best Value for Money 1.5 Ton (Highest Cooling Power - 19K BTU)"

final_comparison['Market_Positioning'] = final_comparison.apply(generate_recommendation, axis=1)

# Display Complete Analytical Comparison Matrix
display(final_comparison[['Brand', 'Model', 'Price_PKR', 'BTU_Rating', 'Compressor_Warranty_Years', 'Positive_Ratio_%', 'Market_Positioning']])

STEP 4: 3-WAY MODEL COMPARISON ENGINE


,Brand,Model,Price_PKR,BTU_Rating,Compressor_Warranty_Years,Positive_Ratio_%,Market_Positioning
0,Dawlance,Mega T+ Inverter 10 CO,78999,9000,12,NaN,Best Choice for Small Bedrooms & Lowest Electr...
1,Haier,HSU-19LF (White),134999,19000,10,NaN,Best Value for Money 1.5 Ton (Highest Cooling ...
2,Haier,Pearl Black (HSU-18HFP),174900,18000,10,NaN,Best Luxury Choice (Smart App Control & Black ...


Dynamic Recommendation Engine & Web Master Export

In [5]:
print("="*60)
print("STEP 5: WEBSITE RECOMMENDATION ALGORITHM & MASTER EXPORT")
print("="*60)

# 1. Recommendation Function for Website User
def recommend_ac_model(room_sqft, budget_pkr):
    print(f"\n[USER QUERY] Room Size: {room_sqft} SqFt | Max Budget: Rs. {budget_pkr:,}")
    matches = df_specs_comparison[
        (df_specs_comparison['Ideal_Room_SqFt'] >= room_sqft) & 
        (df_specs_comparison['Price_PKR'] <= budget_pkr)
    ]
    if not matches.empty:
        print("-> RECOMMENDED AC MODEL(S):")
        for idx, r in matches.iterrows():
            print(f"   * {r['Brand']} {r['Model']} (Price: Rs. {r['Price_PKR']:,}) - {r['Key_Feature']}")
    else:
        print("-> No exact match found within this budget. Consider increasing budget.")

# Test Recommendation Engine
recommend_ac_model(room_sqft=120, budget_pkr=90000)
recommend_ac_model(room_sqft=200, budget_pkr=180000)

# 2. Export Master Payload JSON for Website Import
master_export = []
for idx, row in df_specs_comparison.iterrows():
    p_name = row['Model']
    top_reviews = reviews_master[reviews_master['Product'] == p_name].head(5).to_dict(orient='records')
    
    item = {
        "sku": row['Product_ID'],
        "title": f"{row['Brand']} {row['Model']}",
        "price": row['Price_PKR'],
        "capacity": {
            "ton": row['Capacity_Ton'],
            "btu": row['BTU_Rating'],
            "room_sqft": row['Ideal_Room_SqFt']
        },
        "specifications": {
            "inverter": row['Inverter_Technology'],
            "condenser": row['Condenser_Type'],
            "compressor_warranty": f"{row['Compressor_Warranty_Years']} Years"
        },
        "highlights": row['Key_Feature'],
        "customer_feedback": top_reviews
    }
    master_export.append(item)

with open("master_website_import.json", "w", encoding="utf-8") as f:
    json.dump(master_export, f, indent=4, ensure_ascii=False)

print("\n" + "="*60)
print("[COMPLETED] All Models Executed Successfully!")
print("Files Generated in Working Directory:")
print("1. standardized_products_matrix.csv")
print("2. cleaned_sentiment_reviews.csv")
print("3. master_website_import.json")
print("="*60)

STEP 5: WEBSITE RECOMMENDATION ALGORITHM & MASTER EXPORT

[USER QUERY] Room Size: 120 SqFt | Max Budget: Rs. 90,000
-> RECOMMENDED AC MODEL(S):
   * Dawlance Mega T+ Inverter 10 CO (Price: Rs. 78,999) - 4 Gen Energy Saver & i-Clean

[USER QUERY] Room Size: 200 SqFt | Max Budget: Rs. 180,000
-> RECOMMENDED AC MODEL(S):
   * Haier HSU-19LF (White) (Price: Rs. 134,999) - 19,000 Full BTU & Self Cleaning
   * Haier Pearl Black (HSU-18HFP) (Price: Rs. 174,900) - Black Glass Panel, WiFi & Low Amp (2.4A)

[COMPLETED] All Models Executed Successfully!
Files Generated in Working Directory:
1. standardized_products_matrix.csv
2. cleaned_sentiment_reviews.csv
3. master_website_import.json
